<a href="https://colab.research.google.com/github/harsh3-web/Ecommerce-Recommender/blob/main/01_data_generation_and_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Fresh notebook = fresh computer -> download the repo again and enter its folder
!git clone https://github.com/harsh3-web/Ecommerce-Recommender.git
%cd Ecommerce-Recommender

# model.py needs these to import (PyTorch, torchvision, pandas are already in Colab)
!pip install -q torch_geometric sentence-transformers flask

Cloning into 'Ecommerce-Recommender'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 14 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 11.27 KiB | 195.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/Ecommerce-Recommender/Ecommerce-Recommender


In [11]:
from google.colab import drive
drive.mount("/content/drive")     # connect your Google Drive (log in when asked)

# Copy the 6 CSVs from Drive into the current folder ("." = here, the project folder)
!cp /content/drive/MyDrive/Ecommerce-Recommender-data/*.csv .
!ls -lh *.csv                     # should list 6 files

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw------- 1 root root 157K Oct  1 14:01 browsing_history_expanded.csv
-rw------- 1 root root 8.9K Oct  1 14:01 ground_truth_tastes.csv
-rw------- 1 root root 6.9K Oct  1 14:01 product_images_expanded.csv
-rw------- 1 root root  20K Oct  1 14:01 products_expanded.csv
-rw------- 1 root root  53K Oct  1 14:01 purchases_expanded.csv
-rw------- 1 root root 8.1K Oct  1 14:01 users_expanded.csv


In [12]:
import pandas as pd
import numpy as np


users     = pd.read_csv("users_expanded.csv")
products  = pd.read_csv("products_expanded.csv")
purchases = pd.read_csv("purchases_expanded.csv", parse_dates=["timestamp"])
browsing  = pd.read_csv("browsing_history_expanded.csv", parse_dates=["timestamp"])
tastes    = pd.read_csv("ground_truth_tastes.csv").set_index("user_id")   # user_id becomes the row label

In [13]:
n_users, n_products = len(users), len(products)   # len(table) = number of rows

print(f"Users: {n_users} | Products: {n_products}")
print(f"Browsing events: {len(browsing)} | Purchases: {len(purchases)}")

Users: 500 | Products: 200
Browsing events: 5877 | Purchases: 1845


In [14]:
##Sparsity

interactions = pd.concat([browsing[["user_id", "product_id"]],
                          purchases[["user_id", "product_id"]]]).drop_duplicates()

total_cells = n_users * n_products                 # size of the full user x product matrix
sparsity = 1 - len(interactions) / total_cells     # fraction of the matrix that is EMPTY
print(f"Interactions: {len(interactions)} out of {total_cells} possible")
print(f"Sparsity: {sparsity:.2%}")                 # :.2% -> show as percentage, 2 decimals

Interactions: 5877 out of 100000 possible
Sparsity: 94.12%


In [15]:
# Activity per user
print("Products browsed per user:")
print(browsing.groupby("user_id").size().describe().round(1))

print("\nPurchases per user:")
print(purchases.groupby("user_id").size().describe().round(1))

Products browsed per user:
count    500.0
mean      11.8
std        2.9
min        5.0
25%       10.0
50%       12.0
75%       14.0
max       21.0
dtype: float64

Purchases per user:
count    500.0
mean       3.7
std        1.7
min        1.0
25%        2.8
50%        4.0
75%        5.0
max       11.0
dtype: float64


In [16]:
# Conversion rate
conversion = len(purchases) / len(browsing)
print(f"Browse -> purchase conversion: {conversion:.1%}")

Browse -> purchase conversion: 31.4%


In [17]:
##Category balance and bestsellers
print(products["category"].value_counts())        # products per category -> 25 each

top = purchases["product_id"].value_counts().head(10)          # 10 most-bought product ids
top_table = products.set_index("product_id").loc[top.index, ["product_name", "category"]]
top_table["times_bought"] = top.values             # add the purchase counts as a column
top_table

category
T-Shirts    25
Shirts      25
Jeans       25
Dresses     25
Jackets     25
Shoes       25
Watches     25
Bags        25
Name: count, dtype: int64


,product_name,category,times_bought
product_id,,,
142,Grey Chronograph Shoes,Shoes,62
133,Grey Puffer Jacket,Jackets,55
19,Maroon Relaxed Jeans,Jeans,52
122,Olive Formal Shirt,Shirts,49
10,White Checked Shirt,Shirts,38
177,Red Basic T-Shirt,T-Shirts,35
164,Maroon A-Line Dress,Dresses,34
94,Brown Analog Shoes,Shoes,33
144,Brown Sling Bag,Bags,29


In [18]:
## Long tail and cold-start products

counts = purchases["product_id"].value_counts()          # purchases per product, sorted high -> low
top_20pct = int(0.2 * n_products)                        # 20% of 200 = 40 products
share = counts.head(top_20pct).sum() / len(purchases)    # their share of ALL purchases
print(f"Top 20% of products = {share:.0%} of purchases  (long tail)")

never_bought = n_products - purchases["product_id"].nunique()   # nunique = number of distinct values
print(f"Products never purchased: {never_bought}  (cold-start items)")

Top 20% of products = 51% of purchases  (long tail)
Products never purchased: 2  (cold-start items)


In [20]:
##Importing the recommenders from model.py


import logging

from model import collaborative_filtering, content_based_filtering, hybrid_recommendation

# model.py prints LOTS of debug messages; only show warnings and errors from now on
logging.getLogger().setLevel(logging.WARNING)
print("Imported OK")

Imported OK


In [22]:
##Pick one user and see their real taste

uid = 7
print("Hidden taste:", tastes.loc[uid].tolist())

bought_ids = purchases[purchases["user_id"] == uid]["product_id"]
bought = products[products["product_id"].isin(bought_ids)]
print("Bought:", bought[["product_name", "category"]].values.tolist())

Hidden taste: ['Jeans', 'Jackets']
Bought: [['Maroon Relaxed Jeans', 'Jeans'], ['Grey Puffer Jacket', 'Jackets'], ['Grey Puffer Jacket', 'Jackets']]


In [23]:
##Collaborative filtering for this user

cols = ["product_id", "product_name", "category", "rating", "score"]   # columns to display

cf = collaborative_filtering(uid, purchases, products)   # already sorted by score (high -> low)
cf[cols].head(5)                                         # top 5 recommendations

,product_id,product_name,category,rating,score
121,122,Olive Formal Shirt,Shirts,3.9,0.866667
141,142,Grey Chronograph Shoes,Shoes,4.2,0.855556
186,187,Grey Straight Jeans,Jeans,3.5,0.583333
26,27,Blue Straight Jeans,Jeans,3.9,0.505556
172,173,Navy Denim Jacket,Jackets,3.7,0.479630


In [24]:
#Content-based for this user

cb = content_based_filtering(uid, purchases, browsing, products)
cb = cb.sort_values("score", ascending=False)
cb[cols].head(5)

,product_id,product_name,category,rating,score
116,117,White Bomber Jacket,Jackets,5.0,3.963636
130,131,White Bootcut Jeans,Jeans,5.0,3.963636
152,153,Brown Polo T-Shirt,T-Shirts,4.9,3.884364
79,80,Navy Tote Bag,Bags,4.9,3.884364
168,169,Brown Polo T-Shirt,T-Shirts,4.8,3.805091


In [25]:
##Hybrid for this user

hy = hybrid_recommendation(uid, purchases, browsing, products)   # merges CF + content-based
hy[cols + ["source"]].head(5)                                     # 'source' = which method it came from

,product_id,product_name,category,rating,score,source
183,117,White Bomber Jacket,Jackets,5.0,3.963636,Content-Based Filtering
189,131,White Bootcut Jeans,Jeans,5.0,3.963636,Content-Based Filtering
198,153,Brown Polo T-Shirt,T-Shirts,4.9,3.884364,Content-Based Filtering
166,80,Navy Tote Bag,Bags,4.9,3.884364,Content-Based Filtering
206,169,Brown Polo T-Shirt,T-Shirts,4.8,3.805091,Content-Based Filtering


In [26]:
##to check taste match

def taste_match(recommend_fn, user_ids):
    """Average % of top-10 recommendations that fall in the user's hidden liked categories."""
    results = []
    for u in user_ids:
        top10 = recommend_fn(u).sort_values("score", ascending=False).head(10)
        if len(top10) > 0:                                          # skip users with no recommendations
            results.append(top10["category"].isin(set(tastes.loc[u])).mean())   # mean of True/False = %
    return np.mean(results)

In [27]:
##Compare the three methods


methods = {
    "Collaborative": lambda u: collaborative_filtering(u, purchases, products),
    "Content-based": lambda u: content_based_filtering(u, purchases, browsing, products),
    "Hybrid":        lambda u: hybrid_recommendation(u, purchases, browsing, products),
}
sample_users = tastes.index[:100]
for name, fn in methods.items():
    print(f"{name:14s}: {taste_match(fn, sample_users):.0%}")
print("Random guess  : 25%   (2 liked out of 8 categories)")

Collaborative : 63%
Content-based : 53%
Hybrid        : 53%
Random guess  : 25%   (2 liked out of 8 categories)
